# HomEns на 20 Newsgroups

Пример гомогенного ансамбля: на каждый экстрактор ключевых слов (YAKE, RAKE, TopicRank) учится своя голова, веса берутся из macro-F1 на валидации, предсказание — взвешенная сумма вероятностей.

Датасет скачивается автоматически через `sklearn.datasets.fetch_20newsgroups`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "examples":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from HomEns import HomEns
from KRSB import RakeExtractor, TfidfEncoder, TopicRankExtractor, YakeExtractor
from KRSB.bank import KeywordBank

In [ ]:
CATEGORIES = ["sci.space", "sci.med", "rec.autos", "talk.politics.misc"]
SAMPLES_PER_CLASS = 80
RANDOM_STATE = 42

data = fetch_20newsgroups(
    subset="all",
    categories=CATEGORIES,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=RANDOM_STATE,
)

texts, labels = [], []
counts = {i: 0 for i in range(len(CATEGORIES))}
for text, label in zip(data.data, data.target):
    if counts[label] >= SAMPLES_PER_CLASS:
        continue
    cleaned = " ".join(text.split())
    if len(cleaned) < 80:
        continue
    texts.append(cleaned)
    labels.append(label)
    counts[label] += 1
    if all(v >= SAMPLES_PER_CLASS for v in counts.values()):
        break

x_train, x_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=RANDOM_STATE, stratify=labels
)
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
)
print(f"train={len(x_train)} val={len(x_val)} test={len(x_test)}")
print("classes:", list(data.target_names))

Извлекаем ключевые фразы тремя методами. `KeywordBank` — общая таблица представлений, которую потом можно передать в любой ансамбль.

In [ ]:
extractors = [
    YakeExtractor(ngram=2),
    RakeExtractor(),
    TopicRankExtractor(),
]

train_bank = KeywordBank.from_extractors(x_train, extractors, top_n=12)
val_bank = KeywordBank.from_extractors(x_val, extractors, top_n=12)
test_bank = KeywordBank.from_extractors(x_test, extractors, top_n=12)

print("YAKE:", train_bank.row(0)["yake"][:5])
print("RAKE:", train_bank.row(0)["rake"][:5])
print("TopicRank:", train_bank.row(0)["topicrank"][:5])

Каждая голова HomEns видит только свой столбец ключевых фраз, кодирует keyword-текст и учит логистическую регрессию. Вес головы `w ∝ 1 / (1 - F1_val)`. Предсказание — взвешенная сумма вероятностей.

В этом примере энкодер — TF-IDF, чтобы всё считалось на CPU без SciBERT. В оригинальных ноутбуках на месте головы — дообученный `AutoModelForSequenceClassification` (`HomEns.finetune.train_base_classifier`).

In [ ]:
model = HomEns(encoder=TfidfEncoder(), seed=RANDOM_STATE)
model.fit(train_bank, y_train, X_val=val_bank, y_val=y_val)
print("weights:", {h.method: round(h.weight, 3) for h in model.heads})
print("val F1:", {h.method: round(h.val_f1, 3) for h in model.heads})

pred = model.predict(test_bank)
print(classification_report(y_test, pred, target_names=data.target_names, digits=3))

combos = model.evaluate_combinations(test_bank, y_test)
combos.head()